In [1]:
#install
#%pip install pandas numpy pgmpy

In [2]:
import pandas as pd
import numpy as np
from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.estimators import BayesianEstimator
from pgmpy.inference import VariableElimination

C:\Users\gouri\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\gouri\AppData\Local\Programs\Python\Python313\Lib\site-packages\pgmpy\estimators\__init__.py:4: FutureWarning: `pgmpy.estimators.StructureScore` is deprecated and will be removed in v1.3.0. Use `pgmpy.structure_score` instead.
  from .StructureScore import (


In [3]:
heartDisease = pd.read_csv("heart.csv")
heartDisease = heartDisease.replace("?", np.nan)

print(heartDisease.head())
print(heartDisease.columns.tolist())

    age  sex   cp  trestbps   chol  fbs  restecg  thalach  exang  oldpeak  \
0  63.0  1.0  1.0     145.0  233.0  1.0      2.0    150.0    0.0      2.3   
1  67.0  1.0  4.0     160.0  286.0  0.0      2.0    108.0    1.0      1.5   
2  67.0  1.0  4.0     120.0  229.0  0.0      2.0    129.0    1.0      2.6   
3  37.0  1.0  3.0     130.0  250.0  0.0      0.0    187.0    0.0      3.5   
4  41.0  0.0  2.0     130.0  204.0  0.0      2.0    172.0    0.0      1.4   

   slope   ca  thal  heartdisease  
0    3.0  0.0   6.0             0  
1    2.0  3.0   3.0             2  
2    2.0  2.0   7.0             1  
3    3.0  0.0   3.0             0  
4    1.0  0.0   3.0             0  
['age', 'sex', 'cp', 'trestbps', 'chol', 'fbs', 'restecg', 'thalach', 'exang', 'oldpeak', 'slope', 'ca', 'thal', 'heartdisease']


In [4]:
heartDisease = heartDisease.dropna().copy()

for col in heartDisease.columns:
    heartDisease[col] = pd.to_numeric(
        heartDisease[col], errors="coerce"
    )

heartDisease = heartDisease.dropna().copy()

if "HeartDisease" in heartDisease.columns:
    heartDisease.rename(
        columns={"HeartDisease": "heartdisease"}, inplace=True
    )
elif "target" in heartDisease.columns:
    heartDisease.rename(
        columns={"target": "heartdisease"}, inplace=True
    )

print("Dataset shape:", heartDisease.shape)
print(heartDisease.head())

Dataset shape: (297, 14)
    age  sex   cp  trestbps   chol  fbs  restecg  thalach  exang  oldpeak  \
0  63.0  1.0  1.0     145.0  233.0  1.0      2.0    150.0    0.0      2.3   
1  67.0  1.0  4.0     160.0  286.0  0.0      2.0    108.0    1.0      1.5   
2  67.0  1.0  4.0     120.0  229.0  0.0      2.0    129.0    1.0      2.6   
3  37.0  1.0  3.0     130.0  250.0  0.0      0.0    187.0    0.0      3.5   
4  41.0  0.0  2.0     130.0  204.0  0.0      2.0    172.0    0.0      1.4   

   slope   ca  thal  heartdisease  
0    3.0  0.0   6.0             0  
1    2.0  3.0   3.0             2  
2    2.0  2.0   7.0             1  
3    3.0  0.0   3.0             0  
4    1.0  0.0   3.0             0  


In [5]:
continuous = ["age", "trestbps", "chol", "thalach"]

for col in continuous:
    heartDisease[col] = pd.cut(
        heartDisease[col],
        bins=4,
        labels=False,
        include_lowest=True
    ).astype(int)

print(heartDisease.head())

   age  sex   cp  trestbps  chol  fbs  restecg  thalach  exang  oldpeak  \
0    2  1.0  1.0         1     0  1.0      2.0        2    0.0      2.3   
1    3  1.0  4.0         2     1  0.0      2.0        1    1.0      1.5   
2    3  1.0  4.0         0     0  0.0      2.0        1    1.0      2.6   
3    0  1.0  3.0         1     1  0.0      0.0        3    0.0      3.5   
4    0  0.0  2.0         1     0  0.0      2.0        3    0.0      1.4   

   slope   ca  thal  heartdisease  
0    3.0  0.0   6.0             0  
1    2.0  3.0   3.0             2  
2    2.0  2.0   7.0             1  
3    3.0  0.0   3.0             0  
4    1.0  0.0   3.0             0  


In [6]:

# Construct Bayesian Network
model = DiscreteBayesianNetwork([
    ("age", "trestbps"),
    ("age", "fbs"),
    ("sex", "trestbps"),
    ("exang", "trestbps"),
    ("trestbps", "heartdisease"),
    ("fbs", "heartdisease"),
    ("heartdisease", "restecg"),
    ("heartdisease", "thalach"),
    ("heartdisease", "chol")
])

# Learn Conditional Probability Distributions
estimator = BayesianEstimator(model, heartDisease)

cpds = estimator.get_parameters(
    prior_type="BDeu",
    equivalent_sample_size=5
)

model.add_cpds(*cpds)

print("Bayesian Network trained successfully.")
print("Model valid:", model.check_model())

C:\Users\gouri\AppData\Local\Temp\ipykernel_12632\742893999.py:15: FutureWarning: `pgmpy.estimators.BayesianEstimator` is deprecated and will be removed in v1.3.0. Please use `pgmpy.parameter_estimator.DiscreteBayesianEstimator` instead.
  estimator = BayesianEstimator(model, heartDisease)


Bayesian Network trained successfully.
Model valid: True


In [7]:
infer = VariableElimination(model)

print("1. Probability of Heart Disease given Age category = 1")

q1 = infer.query(
    variables=["heartdisease"],
    evidence={"age": 1}
)

print(q1)

print("\n2. Probability of Heart Disease given Cholesterol category = 1")

q2 = infer.query(
    variables=["heartdisease"],
    evidence={"chol": 1}
)

print(q2)

1. Probability of Heart Disease given Age category = 1
+-----------------+---------------------+
| heartdisease    |   phi(heartdisease) |
+=================+=====================+
| heartdisease(0) |              0.5583 |
+-----------------+---------------------+
| heartdisease(1) |              0.1741 |
+-----------------+---------------------+
| heartdisease(2) |              0.1197 |
+-----------------+---------------------+
| heartdisease(3) |              0.1088 |
+-----------------+---------------------+
| heartdisease(4) |              0.0391 |
+-----------------+---------------------+

2. Probability of Heart Disease given Cholesterol category = 1
+-----------------+---------------------+
| heartdisease    |   phi(heartdisease) |
+=================+=====================+
| heartdisease(0) |              0.4844 |
+-----------------+---------------------+
| heartdisease(1) |              0.2159 |
+-----------------+---------------------+
| heartdisease(2) |              0.1449 |